# ITAP ML Pipeline v4: OSINT-Aligned Threat Predictor
This notebook generates a synthetic organizational-scale dataset based on **real-world OSINT features**, bridging the gap between external attack surface scanning and deep learning.

It trains:
1. **LSTM Exploit Predictor**: Predicts specific attack vectors (RCE, SQLi, etc.) based on 20 OSINT features.
2. **Autoencoder Anomaly Detector**: Detects zero-day risk by calculating the reconstruction error of the OSINT profile.


In [ ]:
!pip install tensorflow scikit-learn numpy pandas tqdm


In [ ]:
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

CKPT_DIR = '/kaggle/working/checkpoints'
WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('Directories created successfully.')


## Section 1: Synthetic OSINT Dataset Generation (1M Records)


In [ ]:
print("Generating 1,000,000 Synthetic OSINT Records...")

# Attack Categories matching ITAP backend
attack_categories = [
    "Normal",                 # 0
    "Remote Code Execution",  # 1
    "SQL Injection",          # 2
    "Cross-Site Scripting",   # 3
    "Privilege Escalation",   # 4
    "Buffer Overflow",        # 5
    "Authentication Bypass",  # 6
    "Denial of Service",      # 7
    "Command Injection",      # 8
    "Directory Traversal",    # 9
    "Server-Side Request Forgery", # 10
    "XML External Entity",    # 11
    "Insecure Deserialization", # 12
    "Information Disclosure"  # 13
]

NUM_RECORDS = 1000000
NUM_FEATURES = 20

X = np.zeros((NUM_RECORDS, NUM_FEATURES), dtype=np.float32)
y = np.zeros((NUM_RECORDS,), dtype=np.int32)

np.random.seed(42)

for i in tqdm(range(NUM_RECORDS)):
    # Base probability for being malicious
    is_malicious = np.random.rand() > 0.4
    
    if is_malicious:
        label = np.random.randint(1, 14)
        
        # High risk features
        max_cvss = np.random.uniform(0.6, 1.0)
        vt_hits = np.random.uniform(0.1, 1.0)
        otx_pulses = np.random.uniform(0.1, 0.8)
        
        # Attack-specific traits
        if label == 1: # RCE
            open_ports = np.random.uniform(0.2, 1.0)
            has_445 = 1.0
            has_3389 = np.random.choice([0.0, 1.0], p=[0.2, 0.8])
        elif label == 2: # SQLi
            open_ports = np.random.uniform(0.0, 0.3)
            has_445 = 0.0
            has_3389 = 0.0
            max_cvss = np.random.uniform(0.7, 1.0) # Often high for SQLi
        else:
            open_ports = np.random.uniform(0.0, 1.0)
            has_445 = np.random.choice([0.0, 1.0])
            has_3389 = np.random.choice([0.0, 1.0])
            
        cve_count = np.random.uniform(0.1, 1.0)
    else:
        label = 0 # Normal
        max_cvss = np.random.uniform(0.0, 0.3)
        vt_hits = 0.0
        otx_pulses = np.random.uniform(0.0, 0.1)
        open_ports = np.random.uniform(0.0, 0.2)
        has_445 = 0.0
        has_3389 = 0.0
        cve_count = np.random.uniform(0.0, 0.1)
        
    X[i, 0] = open_ports          # open_ports_count
    X[i, 1] = np.random.choice([0.0, 1.0]) # has_port_22
    X[i, 2] = has_3389            # has_port_3389
    X[i, 3] = has_445             # has_port_445
    X[i, 4] = 1.0                 # has_port_80_443
    X[i, 5] = max_cvss            # max_cvss
    X[i, 6] = max_cvss * 0.7      # avg_cvss
    X[i, 7] = cve_count           # cve_count
    X[i, 8] = cve_count * 0.4     # exploitable_cve_count
    X[i, 9] = vt_hits             # vt_malicious_hits
    X[i, 10] = vt_hits * 1.2      # vt_suspicious_hits (capped below)
    X[i, 11] = otx_pulses         # otx_pulse_count
    X[i, 12] = np.random.choice([0.0, 1.0], p=[0.9, 0.1]) # is_high_risk_country
    X[i, 13] = open_ports         # shodan_services_count
    X[i, 14] = np.random.choice([0.0, 1.0]) # has_cms
    X[i, 15] = np.random.choice([0.0, 1.0], p=[0.8, 0.2]) # ssl_expired
    X[i, 16] = np.random.uniform(0.0, 1.0) # domain_age_days
    # 17-19 are pads (0.0)
    
    y[i] = label

# Clip to [0, 1]
X = np.clip(X, 0.0, 1.0)

print(f"Generated X shape: {X.shape}, y shape: {y.shape}")

# Prepare for LSTM (samples, timesteps, features)
X_lstm = X.reshape((X.shape[0], 1, X.shape[1]))
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


## Section 2: Train LSTM Exploit Predictor


In [ ]:
LSTM_FINAL_PATH = f"{WEIGHTS_DIR}/itap_lstm_v3.h5"
num_classes = len(attack_categories)

lstm_model = Sequential([
    LSTM(128, activation='relu', input_shape=(1, NUM_FEATURES), return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(num_classes, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print("\nStarting LSTM Training...")
lstm_model.fit(
    X_train, y_train,
    epochs=20, # Reduced for synthetic demo speed
    batch_size=512,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

loss, acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"\nLSTM Final Test Accuracy: {acc*100:.2f}%")
lstm_model.save(LSTM_FINAL_PATH)


## Section 3: Train Autoencoder Anomaly Detector


In [ ]:
AE_FINAL_PATH = f"{WEIGHTS_DIR}/itap_autoencoder_v3.h5"

print("Extracting Normal OSINT profiles for Autoencoder...")
X_normal = X[y == 0]
X_ae_train, X_ae_test = train_test_split(X_normal, test_size=0.2, random_state=123)

input_layer = Input(shape=(NUM_FEATURES,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = BatchNormalization()(encoded)
encoded = Dense(8, activation='relu')(encoded)

decoded = Dense(16, activation='relu')(encoded)
decoded = BatchNormalization()(decoded)
decoded = Dense(NUM_FEATURES, activation='sigmoid')(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

callbacks_ae = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print("\nStarting Autoencoder Training...")
autoencoder.fit(
    X_ae_train, X_ae_train,
    epochs=20,
    batch_size=512,
    validation_data=(X_ae_test, X_ae_test),
    callbacks=callbacks_ae,
    verbose=1
)

loss = autoencoder.evaluate(X_ae_test, X_ae_test, verbose=0)
print(f"\nAutoencoder Final MSE Loss: {loss:.6f}")
autoencoder.save(AE_FINAL_PATH)


## Section 4: Export metadata


In [ ]:
metadata = {
    'lstm_accuracy': float(acc),
    'autoencoder_val_loss': float(loss),
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'features': NUM_FEATURES,
    'type': 'osint_aligned'
}

with open(f'{WEIGHTS_DIR}/model_metadata_v3.json', 'w') as mf:
    json.dump(metadata, mf, indent=2)

print("=== ITAP v4 OSINT MODELS TRAINED SUCCESSFULLY ===")
print("Models saved in /kaggle/working/weights. You can download them now.")
